# Graph Neural Networks — Theory Notebook

> *"A graph neural network is a machine that reads a graph and learns by listening to its neighbors."*

**Companion:** [notes.md](notes.md)

---

## Synthetic Default Contagion Example

A small firm network. Node features encode fundamentals; edges encode exposures. We simulate a simple message-passing step that mixes local risk with neighbor risk (toy version of GATv2-style aggregation). See notes.md §12.6.

In [ ]:
# === Synthetic Contagion Example ===
import numpy as np
np.random.seed(42)

# 6 firms, undirected exposure graph
n = 6
edges = [(0,1), (1,2), (2,3), (3,4), (4,5), (1,4), (2,5)]
A = np.zeros((n, n))
for u, v in edges:
    A[u, v] = A[v, u] = 1.0

# Node features: [leverage, size_log, sector_dummy]
X = np.array([
    [2.1, 4.5, 0],  # firm 0 healthy
    [3.8, 5.1, 1],  # firm 1 higher leverage
    [1.9, 3.8, 0],
    [4.2, 6.0, 1],  # firm 3 stressed
    [2.5, 4.2, 0],
    [5.1, 4.9, 1],  # firm 5 high risk
])

print('Adjacency:')
print(A.astype(int))
print('\nNode features (leverage, log-size, sector):')
print(X)

# Simple normalized aggregation (toy GCN-style)
A_tilde = A + np.eye(n)
D_inv_sqrt = np.diag(1.0 / np.sqrt(A_tilde.sum(axis=1)))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt

# One message-passing step: mix own features with neighbors
H = A_hat @ X
print('\nAfter one aggregation step (H = A_hat @ X):')
print(H.round(3))

# Toy risk score: linear combination emphasizing leverage + contagion
risk = 0.7 * H[:, 0] + 0.3 * H[:, 1]
print('\nToy risk scores (higher = more contagious risk):')
for i, r in enumerate(risk):
    print(f'  Firm {i}: {r:.3f}')

print('\nPASS - synthetic contagion example runs')
print('Takeaway: firm 5 and firm 3 remain high-risk; neighbors of stressed firms inherit elevated scores via message passing.')